In [1]:
"""
Heart Disease Prediction using XGBoost + ADASYN SMOTE
======================================================
Implementation based on:
  "A Novel Heart Disease Prediction System Using XGBoost Classifier
   Coupled With ADASYN SMOTE" (ICCCIS 2023)

Target dataset : df_clean.csv  (Framingham CHD dataset, pre-cleaned)
Target column  : TenYearCHD  (0 = no CHD, 1 = CHD within 10 years)

Run:
    pip install xgboost imbalanced-learn scikit-learn pandas matplotlib seaborn
    python xgboost_heart_disease.py
"""

# ── 0. Imports ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score,
    classification_report, roc_curve
)

from imblearn.combine import SMOTEENN          # ADASYN SMOTE equivalent
from imblearn.over_sampling import ADASYN, SMOTE

import xgboost as xgb

ModuleNotFoundError: No module named 'pandas'

In [4]:
# ── 1. Load Data ──────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Loading Data")
print("=" * 60)

DATA_PATH = "df_clean.csv"     # cleaned data file
TARGET    = "TenYearCHD"       # 10 year prediction

df = pd.read_csv(DATA_PATH)
print(f"Shape : {df.shape}")
print(f"Columns : {list(df.columns)}\n")
print(df.head())

STEP 1 — Loading Data
Shape : (4238, 16)
Columns : ['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'TenYearCHD']

   male  age  education  currentSmoker  cigsPerDay  BPMeds  prevalentStroke  \
0     1   39          4              0         0.0       0                0   
1     0   46          2              0         0.0       0                0   
2     1   48          1              1        20.0       0                0   
3     0   61          3              1        30.0       0                0   
4     0   46          3              1        23.0       0                0   

   prevalentHyp  diabetes  totChol  sysBP  diaBP    BMI  heartRate  glucose  \
0             0         0    195.0  106.0   70.0  26.97       80.0     77.0   
1             0         0    250.0  121.0   81.0  28.73       95.0     76.0   
2             0         0    245.0  127.5   8

In [5]:
# ── 2. Exploratory: Class Distribution ───────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Class Distribution (before balancing)")
print("=" * 60)

class_counts = df[TARGET].value_counts()
print(class_counts)
print(f"\nClass imbalance ratio: {class_counts[0]/class_counts[1]:.2f}:1")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of class distribution
axes[0].bar(["No CHD (0)", "CHD (1)"], class_counts.values,
            color=["steelblue", "tomato"], edgecolor="black")
axes[0].set_title("Class Distribution (Before Balancing)")
axes[0].set_ylabel("Count")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 20, f"{v}\n({v/len(df)*100:.1f}%)",
                 ha="center", fontsize=10)

# Correlation heatmap (like Fig. 2 in the paper)
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=axes[1], cmap="coolwarm", center=0,
            annot=False, mask=mask, linewidths=0.3)
axes[1].set_title("Feature Correlation Heatmap")

plt.tight_layout()
plt.savefig("01_eda.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → 01_eda.png")


STEP 2 — Class Distribution (before balancing)
TenYearCHD
0    3594
1     644
Name: count, dtype: int64

Class imbalance ratio: 5.58:1
Saved → 01_eda.png


In [6]:
# ── 3. Preprocessing ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — Preprocessing")
print("=" * 60)

# 3a. Separate features / target
X = df.drop(columns=[TARGET])
y = df[TARGET]

# 3b. Encode any remaining categorical columns (should be none in df_clean,
#     but handle gracefully)
le = LabelEncoder()
for col in X.select_dtypes(include=["object", "category"]).columns:
    X[col] = le.fit_transform(X[col].astype(str))
    print(f"  Label-encoded: {col}")

# 3c. Standard Scaling (same as the paper's StandardScaler usage)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Features after preprocessing: {X_scaled.shape[1]}")
print(f"Missing values: {X_scaled.isnull().sum().sum()}")


STEP 3 — Preprocessing
Features after preprocessing: 15
Missing values: 0


In [7]:
# ── 4. Train / Test Split (80:20 as requested) ────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Train/Test Split  (80 : 20)")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=42,
    stratify=y           # preserve class ratio in both splits
)
print(f"Train set : {X_train.shape}  |  Test set : {X_test.shape}")
print(f"Train CHD prevalence : {y_train.mean():.3f}")
print(f"Test  CHD prevalence : {y_test.mean():.3f}")


STEP 4 — Train/Test Split  (80 : 20)
Train set : (3390, 15)  |  Test set : (848, 15)
Train CHD prevalence : 0.152
Test  CHD prevalence : 0.152


In [8]:
# ── 5. Class Balancing with ADASYN SMOTE ─────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — ADASYN SMOTE Balancing (applied to TRAIN only)")
print("=" * 60)

# Paper uses ADASYN SMOTE = ADASYN applied after SMOTE synthetic samples
# imblearn's ADASYN is the direct equivalent used in the paper.
# We also compare plain SMOTE to reproduce Fig. 7 of the paper.

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"After SMOTE       : {pd.Series(y_train_smote).value_counts().to_dict()}")

adasyn_smote = ADASYN(random_state=42)
X_train_bal, y_train_bal = adasyn_smote.fit_resample(X_train, y_train)
print(f"After ADASYN SMOTE: {pd.Series(y_train_bal).value_counts().to_dict()}")

# Visualise balancing
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (data, title) in zip(axes, [
    (y_train,       "Original Train"),
    (y_train_smote, "After SMOTE"),
    (y_train_bal,   "After ADASYN SMOTE"),
]):
    counts = pd.Series(data).value_counts()
    ax.pie(counts.values, labels=["No CHD", "CHD"],
           autopct="%1.1f%%", colors=["steelblue", "tomato"],
           startangle=90)
    ax.set_title(title)

plt.tight_layout()
plt.savefig("02_balancing.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → 02_balancing.png")


STEP 5 — ADASYN SMOTE Balancing (applied to TRAIN only)
After SMOTE       : {0: 2875, 1: 2875}
After ADASYN SMOTE: {1: 2987, 0: 2875}
Saved → 02_balancing.png


In [9]:
# ── 6. XGBoost Model — Paper's Proposed Classifier (Baseline) ────────────────
print("\n" + "=" * 60)
print("STEP 6 — XGBoost Classifier (ADASYN SMOTE data) - Baseline")
print("=" * 60)

xgb_paper_model = xgb.XGBClassifier(
    n_estimators=200,         # ensemble of boosted trees
    max_depth=6,              # tree depth
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_paper_model.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test, y_test)],
    verbose=False
)
print("Paper's XGBoost model training complete.")


STEP 6 — XGBoost Classifier (ADASYN SMOTE data) - Baseline
Paper's XGBoost model training complete.


In [ ]:
# ── 7. XGBoost Model — Hyperparameter Tuning with Grid Search & CV ──────────
print("\n" + "=" * 60)
print("STEP 7 — XGBoost Hyperparameter Tuning (Grid Search + CV)")
print("=" * 60)

from sklearn.model_selection import GridSearchCV

# Define the parameter grid (simplified for demonstration, can be expanded)
# The original paper did not specify these, so these are common choices.
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

# Initialize XGBoost Classifier
xgb_clf = xgb.XGBClassifier(
    use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1
)

# Setup GridSearchCV
# Using StratifiedKFold to maintain class distribution in folds
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(estimator=xgb_clf, param_grid=param_grid,
                           scoring='f1', cv=stratified_kfold, verbose=0, n_jobs=-1)

grid_search.fit(X_train_bal, y_train_bal)

print("Best hyperparameters found:", grid_search.best_params_)

xgb_optimized_model = grid_search.best_estimator_

# Cross-validation scores (F1-score) for the best model
cv_results = grid_search.cv_results_['mean_test_score'][grid_search.best_index_]
# The following retrieves all F1 scores for the best parameter set across folds
best_params_idx = grid_search.best_index_
best_f1_scores = []
for i in range(stratified_kfold.n_splits):
    best_f1_scores.append(grid_search.cv_results_[f'split{i}_test_score'][best_params_idx])

print(f"Mean Cross-Validation F1-Score: {np.mean(best_f1_scores):.4f}")
print(f"Std Dev of Cross-Validation F1-Score: {np.std(best_f1_scores):.4f}")


STEP 7 — XGBoost Hyperparameter Tuning (Grid Search + CV)


In [ ]:
# ── 8. Evaluation on Held-Out Test Set (Optimized XGBoost) ──────────────────
print("\n" + "=" * 60)
print("STEP 8 — Evaluation on Held-Out Test Set (Optimized XGBoost)")
print("=" * 60)

y_pred_opt      = xgb_optimized_model.predict(X_test)
y_pred_prob_opt = xgb_optimized_model.predict_proba(X_test)[:, 1]

acc_opt  = accuracy_score(y_test, y_pred_opt)  * 100
rec_opt  = recall_score(y_test, y_pred_opt)    * 100
prec_opt = precision_score(y_test, y_pred_opt) * 100
f1_opt   = f1_score(y_test, y_pred_opt)        * 100
auc_opt  = roc_auc_score(y_test, y_pred_prob_opt) * 100

precision_opt, recall_opt, _ = precision_recall_curve(y_test, y_pred_prob_opt)
pr_auc_opt = auc_score(recall_opt, precision_opt) * 100

print(f"\n{'Metric':<20} {'XGBoost + ADASYN SMOTE (Optimized)':>40}")
print("-" * 62)
print(f"{'Accuracy':<20} {acc_opt:>39.2f}%")
print(f"{'Recall':<20} {rec_opt:>39.2f}%")
print(f"{'Precision':<20} {prec_opt:>39.2f}%")
print(f"{'F1-Score':<20} {f1_opt:>39.2f}%")
print(f"{'ROC-AUC':<20} {auc_opt:>39.2f}%")
print(f"{'PR-AUC':<20} {pr_auc_opt:>39.2f}%")

print("\nFull Classification Report (Optimized XGBoost):")
print(classification_report(y_test, y_pred_opt, target_names=["No CHD", "CHD"]))

In [ ]:
# ── 7. Evaluation ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — Evaluation on Held-Out Test Set (Baseline XGBoost)")
print("=" * 60)

y_pred_paper      = xgb_paper_model.predict(X_test)
y_pred_prob_paper = xgb_paper_model.predict_proba(X_test)[:, 1]

acc_paper  = accuracy_score(y_test, y_pred_paper)  * 100
rec_paper  = recall_score(y_test, y_pred_paper)    * 100
prec_paper = precision_score(y_test, y_pred_paper) * 100
f1_paper   = f1_score(y_test, y_pred_paper)        * 100
auc_paper  = roc_auc_score(y_test, y_pred_prob_paper) * 100

from sklearn.metrics import precision_recall_curve, auc as auc_score
precision_paper, recall_paper, _ = precision_recall_curve(y_test, y_pred_prob_paper)
pr_auc_paper = auc_score(recall_paper, precision_paper) * 100

print(f"\n{'Metric':<20} {'XGBoost + ADASYN SMOTE (Baseline)':>35}")
print("-" * 57)
print(f"{'Accuracy':<20} {acc_paper:>34.2f}%")
print(f"{'Recall':<20} {rec_paper:>34.2f}%")
print(f"{'Precision':<20} {prec_paper:>34.2f}%")
print(f"{'F1-Score':<20} {f1_paper:>34.2f}%")
print(f"{'ROC-AUC':<20} {auc_paper:>34.2f}%")
print(f"{'PR-AUC':<20} {pr_auc_paper:>34.2f}%")

print("\nFull Classification Report (Baseline XGBoost):")
print(classification_report(y_test, y_pred_paper, target_names=["No CHD", "CHD"]))

In [ ]:
# ── 9. Comparison: All Six Classifiers (replicating Table 2 of the paper) ────
print("\n" + "=" * 60)
print("STEP 9 — Comparison: Six Classifiers (Paper Table 2 replica)")
print("=" * 60)

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

classifiers = {
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SGD":                 SGDClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost":            AdaBoostClassifier(n_estimators=100, random_state=42),
    "XGBoost (Baseline)":  xgb_paper_model,          # already trained — we re-fit for fairness
    "XGBoost (Optimized)": xgb_optimized_model,      # optimized model
}

results = {}
for name, clf in classifiers.items():
    # For the already trained models (baseline and optimized XGBoost), skip refitting.
    if name == "XGBoost (Baseline)":
        preds = xgb_paper_model.predict(X_test)
    elif name == "XGBoost (Optimized)":
        preds = xgb_optimized_model.predict(X_test)
    else:
        clf.fit(X_train_bal, y_train_bal)
        preds = clf.predict(X_test)

    results[name] = {
        "Accuracy (%)":  round(accuracy_score(y_test, preds)  * 100, 2),
        "Recall (%)":    round(recall_score(y_test, preds)    * 100, 2),
        "Precision (%)": round(precision_score(y_test, preds) * 100, 2),
        "F1-Score (%)":  round(f1_score(y_test, preds)        * 100, 2),
    }
    print(f"  {name:<22}: Acc={results[name]['Accuracy (%)']:.2f}%  "
          f"Rec={results[name]['Recall (%)']:.2f}%  "
          f"Prec={results[name]['Precision (%)']:.2f}%  "
          f"F1={results[name]['F1-Score (%)']:.2f}%")

results_df = pd.DataFrame(results).T
print("\n", results_df.to_string())

In [ ]:
# ── 10. Figures ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 11)) # Increased figure size and subplots for PR Curve

# Fig A — Confusion Matrix (Optimized XGBoost)
cm_opt = confusion_matrix(y_test, y_pred_opt)
ConfusionMatrixDisplay(cm_opt, display_labels=["No CHD", "CHD"]).plot(
    ax=axes[0, 0], colorbar=False, cmap="Blues"
)
axes[0, 0].set_title("Confusion Matrix — XGBoost + ADASYN SMOTE (Optimized)")

# Fig B — Accuracy comparison bar chart
accs = [v["Accuracy (%)"] for v in results.values()]

# Highlight the optimized XGBoost model
num_classifiers = len(results)
colors = ["#4C72B0"] * (num_classifiers - 1) + ["#DD8452"]
bars = axes[0, 1].bar(results.keys(), accs, color=colors, edgecolor="black")
axes[0, 1].set_ylim(min(accs) - 2, max(accs) + 2)
axes[0, 1].set_title("Accuracy Comparison of Classifiers")
axes[0, 1].set_ylabel("Accuracy (%)")
axes[0, 1].tick_params(axis="x", rotation=45)
for bar, val in zip(bars, accs):
    axes[0, 1].text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.1, f"{val:.2f}%",
                    ha="center", va="bottom", fontsize=8)

# Fig C — SMOTE vs ADASYN SMOTE accuracy for XGBoost (Baseline vs Optimized)
xgb_smote = xgb.XGBClassifier(n_estimators=200, max_depth=6,
                                learning_rate=0.1, subsample=0.8,
                                colsample_bytree=0.8,
                                use_label_encoder=False,
                                eval_metric="logloss",
                                random_state=42, n_jobs=-1)
xgb_smote.fit(X_train_smote, y_train_smote)
acc_smote    = accuracy_score(y_test, xgb_smote.predict(X_test)) * 100
acc_adasyn_baseline = results["XGBoost (Baseline)"]["Accuracy (%)"]
acc_adasyn_optimized = results["XGBoost (Optimized)"]["Accuracy (%)"]

axes[0, 2].bar(["SMOTE + XGBoost", "ADASYN SMOTE\n+ XGBoost (Baseline)", "ADASYN SMOTE\n+ XGBoost (Optimized)"],
               [acc_smote, acc_adasyn_baseline, acc_adasyn_optimized],
               color=["#4C72B0", "#8CD17D", "#DD8452"], edgecolor="black", width=0.4)
axes[0, 2].set_ylim(min(acc_smote, acc_adasyn_baseline, acc_adasyn_optimized) - 1,
                     max(acc_smote, acc_adasyn_baseline, acc_adasyn_optimized) + 1)
axes[0, 2].set_title("SMOTE vs ADASYN SMOTE — XGBoost Accuracy")
axes[0, 2].set_ylabel("Accuracy (%)")
for i, v in enumerate([acc_smote, acc_adasyn_baseline, acc_adasyn_optimized]):
    axes[0, 2].text(i, v + 0.05, f"{v:.2f}%", ha="center", fontsize=10)

# Fig D — ROC Curve (Optimized XGBoost)
fpr_opt, tpr_opt, _ = roc_curve(y_test, y_pred_prob_opt)
axes[1, 0].plot(fpr_opt, tpr_opt, color="tomato", lw=2,
                label=f"XGBoost (AUC = {auc_opt/100:.3f})")
axes[1, 0].plot([0, 1], [0, 1], "k--", lw=1)
axes[1, 0].set_xlabel("False Positive Rate")
axes[1, 0].set_ylabel("True Positive Rate")
axes[1, 0].set_title("ROC Curve — XGBoost + ADASYN SMOTE (Optimized)")
axes[1, 0].legend(loc="lower right")

# Fig E — Precision-Recall Curve (Optimized XGBoost)
axes[1, 1].plot(recall_opt, precision_opt, color='purple', lw=2,
                label=f'XGBoost (PR-AUC = {pr_auc_opt/100:.3f})')
axes[1, 1].set_xlabel('Recall')
axes[1, 1].set_ylabel('Precision')
axes[1, 1].set_title('Precision-Recall Curve — XGBoost + ADASYN SMOTE (Optimized)')
axes[1, 1].legend(loc='lower left')
axes[1, 1].set_ylim([0.0, 1.05])
axes[1, 1].set_xlim([0.0, 1.0])

# Blank subplot for better layout (optional, or can add another plot)
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig("03_results_optimized.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved → 03_results_optimized.png")

In [ ]:
# ── 11. Feature Importance ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 11 — Feature Importance (Optimized XGBoost)")
print("=" * 60)

importances = pd.Series(
    xgb_optimized_model.feature_importances_, index=X.columns
).sort_values(ascending=False)

print(importances.to_string())

plt.figure(figsize=(8, 5))
importances.plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Feature Importance — XGBoost (Optimized)")
plt.ylabel("Importance Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("04_feature_importance_optimized.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → 04_feature_importance_optimized.png")

In [ ]:
# ── 7. XGBoost Model — Hyperparameter Tuning with Grid Search & CV ──────────
print("\n" + "=" * 60)
print("STEP 7 — XGBoost Hyperparameter Tuning (Grid Search + CV)")
print("=" * 60)

from sklearn.model_selection import GridSearchCV

# Define the parameter grid (simplified for demonstration, can be expanded)
# The original paper did not specify these, so these are common choices.
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

# Initialize XGBoost Classifier
xgb_clf = xgb.XGBClassifier(
    use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1
)

# Setup GridSearchCV
# Using StratifiedKFold to maintain class distribution in folds
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(estimator=xgb_clf, param_grid=param_grid,
                           scoring='f1', cv=stratified_kfold, verbose=0, n_jobs=-1)

grid_search.fit(X_train_bal, y_train_bal)

print("Best hyperparameters found:", grid_search.best_params_)

xgb_optimized_model = grid_search.best_estimator_

# Cross-validation scores (F1-score) for the best model
cv_results = grid_search.cv_results_['mean_test_score'][grid_search.best_index_]
# The following retrieves all F1 scores for the best parameter set across folds
best_params_idx = grid_search.best_index_
best_f1_scores = []
for i in range(stratified_kfold.n_splits):
    best_f1_scores.append(grid_search.cv_results_[f'split{i}_test_score'][best_params_idx])

print(f"Mean Cross-Validation F1-Score: {np.mean(best_f1_scores):.4f}")
print(f"Std Dev of Cross-Validation F1-Score: {np.std(best_f1_scores):.4f}")

In [ ]:
# ── 8. Evaluation on Held-Out Test Set (Optimized XGBoost) ──────────────────
print("\n" + "=" * 60)
print("STEP 8 — Evaluation on Held-Out Test Set (Optimized XGBoost)")
print("=" * 60)

y_pred_opt      = xgb_optimized_model.predict(X_test)
y_pred_prob_opt = xgb_optimized_model.predict_proba(X_test)[:, 1]

acc_opt  = accuracy_score(y_test, y_pred_opt)  * 100
rec_opt  = recall_score(y_test, y_pred_opt)    * 100
prec_opt = precision_score(y_test, y_pred_opt) * 100
f1_opt   = f1_score(y_test, y_pred_opt)        * 100
auc_opt  = roc_auc_score(y_test, y_pred_prob_opt) * 100

precision_opt, recall_opt, _ = precision_recall_curve(y_test, y_pred_prob_opt)
pr_auc_opt = auc_score(recall_opt, precision_opt) * 100

print(f"\n{'Metric':<20} {'XGBoost + ADASYN SMOTE (Optimized)':>40}")
print("-" * 62)
print(f"{'Accuracy':<20} {acc_opt:>39.2f}%")
print(f"{'Recall':<20} {rec_opt:>39.2f}%")
print(f"{'Precision':<20} {prec_opt:>39.2f}%")
print(f"{'F1-Score':<20} {f1_opt:>39.2f}%")
print(f"{'ROC-AUC':<20} {auc_opt:>39.2f}%")
print(f"{'PR-AUC':<20} {pr_auc_opt:>39.2f}%")

print("\nFull Classification Report (Optimized XGBoost):")
print(classification_report(y_test, y_pred_opt, target_names=["No CHD", "CHD"]))

In [ ]:
# ── 12. Save Results Table ────────────────────────────────────────────────────
results_df.to_csv("classifier_comparison_optimized.csv")
print("\nSaved → classifier_comparison_optimized.csv")

print("\n" + "=" * 60)
print("DONE — All outputs saved.")
print("=" * 60)